In [8]:
import os, json, glob, zipfile
import numpy as np
import pandas as pd
from tqdm import tqdm

# paths: change these if your files sit somewhere else
ZIP_INTL = 't20s_male_json (2).zip'
ZIP_IPL  = 'ipl_male_json.zip'
DIR_INTL = 't20_I_json'
DIR_IPL  = 'ipl_json'

# unzip only once, so re-running the cell is cheap
for z, d in [(ZIP_INTL, DIR_INTL), (ZIP_IPL, DIR_IPL)]:
    if not os.path.isdir(d):
        with zipfile.ZipFile(z) as f:
            f.extractall(d)

intl_files = sorted(glob.glob(os.path.join(DIR_INTL, '*.json')))
ipl_files  = sorted(glob.glob(os.path.join(DIR_IPL, '*.json')))
print('intl files:', len(intl_files), '| ipl files:', len(ipl_files))

# dismissal types that count as a bowler's wicket (run outs do not)
BOWLER_WKT = {'bowled', 'caught', 'caught and bowled', 'lbw', 'stumped', 'hit wicket'}
# these do not end the batter's innings as "out"
NOT_OUT = {'retired hurt', 'retired not out'}


def parse(fp, src):
    try:
        d = json.load(open(fp, encoding='utf-8'))
    except Exception:
        return None
    info = d['info']
    if info.get('gender') != 'male' or 'players' not in info:
        return None

    reg = info.get('registry', {}).get('people', {})
    mid = os.path.basename(fp)[:-5]
    date = info['dates'][0]
    teams = info['teams']

    stats = {}

    def get(team, p):
        k = (team, p)
        if k not in stats:
            stats[k] = dict(batted=0, bat_pos=np.nan, runs=0, balls_faced=0, fours=0, sixes=0,
                            dot_faced=0, is_out=0, bat_pp_balls=0, bat_death_balls=0,
                            balls_bowled=0, runs_conceded=0, wickets=0, dot_bowled=0,
                            wides=0, noballs=0, bowl_pp_balls=0, bowl_death_balls=0,
                            catches=0, stumpings=0, run_outs=0)
        return stats[k]

    team_total = {}
    first_bat = None

    for inn in d['innings']:
        if inn.get('super_over'):
            continue
        bat_team = inn['team']
        bowl_team = [t for t in teams if t != bat_team][0]
        if first_bat is None:
            first_bat = bat_team

        order = []          # order in which batters reached the crease
        total = 0
        for ov in inn['overs']:
            over_no = ov['over']
            for dl in ov['deliveries']:
                striker, non_striker, bowler = dl['batter'], dl['non_striker'], dl['bowler']

                # a batter "batted" once he is at the crease, even if he faced no ball
                for x in (striker, non_striker):
                    if x not in order:
                        order.append(x)
                        s = get(bat_team, x)
                        s['batted'] = 1
                        s['bat_pos'] = len(order)

                r = dl['runs']
                ex = dl.get('extras', {})
                total += r['total']
                is_wide = 'wides' in ex
                is_nb = 'noballs' in ex

                sb = get(bat_team, striker)
                if not is_wide:                       # a wide is not a ball faced
                    sb['balls_faced'] += 1
                    if r['batter'] == 0:
                        sb['dot_faced'] += 1
                    if over_no < 6:
                        sb['bat_pp_balls'] += 1
                    if over_no >= 15:
                        sb['bat_death_balls'] += 1
                sb['runs'] += r['batter']
                if r['batter'] == 4:
                    sb['fours'] += 1
                if r['batter'] == 6:
                    sb['sixes'] += 1

                sw = get(bowl_team, bowler)
                # byes and leg byes are not charged to the bowler
                conceded = r['total'] - ex.get('byes', 0) - ex.get('legbyes', 0)
                sw['runs_conceded'] += conceded
                if is_wide:
                    sw['wides'] += 1
                if is_nb:
                    sw['noballs'] += 1
                if not is_wide and not is_nb:
                    sw['balls_bowled'] += 1
                    if conceded == 0:
                        sw['dot_bowled'] += 1
                    if over_no < 6:
                        sw['bowl_pp_balls'] += 1
                    if over_no >= 15:
                        sw['bowl_death_balls'] += 1

                for w in dl.get('wickets', []):
                    kind = w['kind']
                    if kind not in NOT_OUT:
                        get(bat_team, w['player_out'])['is_out'] = 1
                    if kind in BOWLER_WKT:
                        sw['wickets'] += 1
                    for f in w.get('fielders', []):
                        if f.get('substitute'):
                            continue
                        name = f.get('name')
                        if name is None:
                            continue
                        sf = get(bowl_team, name)
                        if kind in ('caught', 'caught and bowled'):
                            sf['catches'] += 1
                        elif kind == 'stumped':
                            sf['stumpings'] += 1
                        elif kind == 'run out':
                            sf['run_outs'] += 1
        team_total[bat_team] = total

    winner = info.get('outcome', {}).get('winner')
    toss = info.get('toss', {})
    rows = []
    for t in teams:
        opp = [x for x in teams if x != t][0]
        innings_no = 1 if t == first_bat else 2
        for p in info['players'].get(t, []):
            row = dict(get(t, p))
            row.update(
                match_id=mid, date=date, venue=info.get('venue'), city=info.get('city'),
                event=(info.get('event') or {}).get('name'), season=str(info.get('season')),
                src=src, team_type=info.get('team_type'),
                team=t, opp=opp, innings=innings_no,
                toss_won=int(toss.get('winner') == t), toss_decision=toss.get('decision'),
                player_id=reg.get(p, p), player=p,
                team_total=team_total.get(t, np.nan), opp_total=team_total.get(opp, np.nan),
                team_won=(np.nan if winner is None else int(winner == t)),
            )
            rows.append(row)
    return rows


rows = []
for fp in tqdm(intl_files, desc='international'):
    r = parse(fp, 'intl')
    if r:
        rows.extend(r)
for fp in tqdm(ipl_files, desc='IPL'):
    r = parse(fp, 'ipl')
    if r:
        rows.extend(r)

df = pd.DataFrame(rows)
df['date'] = pd.to_datetime(df['date'])
df['overs_bowled'] = (df.balls_bowled // 6) + (df.balls_bowled % 6) / 10
df = df.sort_values(['date', 'match_id']).reset_index(drop=True)
df.to_pickle('raw_players.pkl')

print(df.shape, '| matches:', df.match_id.nunique(), '| players:', df.player_id.nunique())
print(df.groupby('src').agg(rows=('match_id', 'size'), matches=('match_id', 'nunique')))
print('date range:', df.date.min().date(), 'to', df.date.max().date())

intl files: 3646 | ipl files: 1243


IPL: 100%|█████████████████████████████████████████████████████████████████████████| 1243/1243 [00:20<00:00, 61.13it/s]


(108131, 40) | matches: 4889 | players: 5166
       rows  matches
src                 
intl  80222     3646
ipl   27909     1243
date range: 2005-02-17 to 2026-10-25


In [2]:
import shutil
shutil.rmtree('t20_I_json', ignore_errors=True)   # old copy, will be re-extracted from the zip

In [6]:
import os, shutil

# make sure the zip is really here before deleting the old folder
print(os.path.exists('ipl_male_json.zip'))

True


In [7]:
shutil.rmtree('ipl_json', ignore_errors=True)   # partial copy with only 222 files

In [9]:
import pandas as pd
import numpy as np

df = pd.read_pickle('raw_players.pkl')
intl = df[df.src == 'intl']
ipl = df[df.src == 'ipl']

# select teams are not countries, so they say nothing about nationality
NON_NATIONAL = {'ICC World XI'}

# set of countries each player has represented in the international data
nat = intl[~intl.team.isin(NON_NATIONAL)].groupby('player_id').team.agg(set)
played_india = {p for p, s in nat.items() if 'India' in s}
played_other = {p for p, s in nat.items() if 'India' not in s}

# cricsheet does not publish Afghanistan matches, so Afghan players show up as
# "IPL only" and would be treated as Indian. A few English players have no
# international rows either. They are added by hand (names as in cricsheet).
MANUAL_FOREIGN = [
    'Rashid Khan', 'Mohammad Nabi', 'Mujeeb Ur Rahman', 'Noor Ahmad', 'Naveen-ul-Haq',
    'Fazalhaq Farooqi', 'Azmatullah Omarzai', 'Rahmanullah Gurbaz', 'Gulbadin Naib',
    'Karim Janat', 'Sediqullah Atal', 'AM Ghazanfar',
    'DA Payne', 'T Kohler-Cadmore',
]

# match names to ids using IPL rows only (some names are shared by unrelated players)
name_to_ids = ipl.groupby('player').player_id.agg(set)
manual_ids = set()
for n in MANUAL_FOREIGN:
    if n in name_to_ids.index:
        manual_ids |= name_to_ids[n]
    else:
        print('name not found in IPL data:', n)

# one row per player
latest_name = df.sort_values('date').groupby('player_id').player.last()
elig = pd.DataFrame({'player': latest_name})
elig['n_intl'] = intl.groupby('player_id').size().reindex(elig.index).fillna(0).astype(int)
elig['n_ipl'] = ipl.groupby('player_id').size().reindex(elig.index).fillna(0).astype(int)
elig['last_ipl'] = ipl.groupby('player_id').date.max().reindex(elig.index)
elig['last_india'] = intl[intl.team == 'India'].groupby('player_id').date.max().reindex(elig.index)

def status(pid):
    if pid in played_india:
        return 'india_capped'
    if pid in played_other or pid in manual_ids:
        return 'foreign'
    return 'uncapped_assumed_indian'   # only IPL rows and not on the manual list

elig['status'] = [status(p) for p in elig.index]
elig['india_eligible'] = elig.status != 'foreign'
elig.index.name = 'player_id'
elig.to_pickle('eligibility.pkl')

print(elig.status.value_counts())

# players active in the 2024+ IPL who are assumed Indian: check this list by eye
recent = ipl[ipl.date >= '2024-01-01'].player_id.unique()
chk = elig.loc[elig.index.isin(recent) & (elig.status == 'uncapped_assumed_indian'), 'player']
print(len(chk), 'assumed-Indian players in IPL since 2024:')
print(', '.join(sorted(chk)))


status
foreign                    4702
uncapped_assumed_indian     339
india_capped                125
Name: count, dtype: int64
112 assumed-Indian players in IPL since 2024:
A Badoni, A Kamboj, A Manohar, A Mhatre, A Raghuvanshi, AA Kulkarni, AS Roy, Abdul Samad, Abhinandan Singh, Abishek Porel, Akash Deep, Akash Madhwal, Akash Singh, Akshat Raghuwanshi, Aniket Verma, Anmolpreet Singh, Anuj Rawat, Arjun Tendulkar, Arshad Khan, Ashutosh Sharma, Ashwani Kumar, Atharva Taide, Auqib Nabi, BR Sharath, Brijesh Sharma, DG Nalkande, DS Rathi, DV Malewar, Gurjapneet Singh, H Sharma, Harpreet Brar, Harsh Dubey, Himmat Singh, K Gowtham, K Kartikeya, K Khejroliya, KK Nair, KR Sen, KS Rathore, Kartik Sharma, Kartik Tyagi, Krish Bhagat, Kumar Kushagra, Lalit Yadav, M Shahrukh Khan, M Siddharth, M Tiwari, MA Agarwal, MD Choudhary, MJ Suthar, MK Lomror, MS Bhandage, Mayank Dagar, Mayank Rawat, Mohsin Khan, Mukesh Choudhary, Musheer Khan, N Sindhu, N Wadhera, P Dubey, P Simran Singh, PP Hinge, PR Veer

In [10]:
import pandas as pd
import numpy as np

df = pd.read_pickle('raw_players.pkl')

# ---- "as-of" rows ----
# one fake row per player dated far in the future. Every feature on that row is
# computed from the player's whole history, so it gives his form "as of" any
# date we ask for (needed for squad members who are not in the actual XI).
ph = df.sort_values('date').groupby('player_id').tail(1).copy()
zero_cols = ['batted', 'runs', 'balls_faced', 'fours', 'sixes', 'dot_faced', 'is_out',
             'bat_pp_balls', 'bat_death_balls', 'balls_bowled', 'runs_conceded', 'wickets',
             'dot_bowled', 'wides', 'noballs', 'bowl_pp_balls', 'bowl_death_balls',
             'catches', 'stumpings', 'run_outs', 'overs_bowled']
for c in zero_cols:
    ph[c] = 0
ph['bat_pos'] = np.nan
ph['date'] = pd.Timestamp('2100-01-01')
ph['match_id'] = 'PHANTOM'
for c in ['team', 'opp', 'team_total', 'opp_total', 'venue']:
    ph[c] = np.nan
ph['src'] = 'intl'            # selection is for an international match
ph['toss_won'] = 0
ph['toss_decision'] = 'field'
ph['innings'] = 1
df = pd.concat([df, ph], ignore_index=True)

df['is_intl'] = (df.src == 'intl').astype(int)
df['bat_first_dec'] = ((df.toss_decision == 'bat') == (df.toss_won == 1)).astype(int)
df = df.sort_values(['player_id', 'date', 'match_id']).reset_index(drop=True)

# priors come from training years only (before 2024)
TRAIN_END = pd.Timestamp('2024-01-01')
tr = df[df.date < TRAIN_END]
bat_cols = ['runs', 'balls_faced', 'fours', 'sixes', 'dot_faced', 'is_out',
            'bat_pos', 'bat_pp_balls', 'bat_death_balls']
bowl_cols = ['balls_bowled', 'runs_conceded', 'wickets', 'dot_bowled', 'wides',
             'noballs', 'bowl_pp_balls', 'bowl_death_balls']
fld_cols = ['catches', 'stumpings', 'run_outs']
prior_bat = tr[tr.batted == 1][bat_cols].mean()
prior_bowl = tr[tr.balls_bowled > 0][bowl_cols].mean()

K = 5      # shrinkage weight: the prior counts as 5 matches
g = df.groupby('player_id')


def roll(mask, cols, W, tag, prior, n_name):
    """Mean of each column over the last W qualifying matches (all if W is None).
    Only rows before the current one are used (shift), and the mean is
    shrunk towards the prior: (S + K*prior) / (N + K)."""
    out = {}
    m = mask.astype(float)
    cm = m.groupby(df.player_id).cumsum()
    for c in cols:
        x = df[c].fillna(0) * m
        cx = x.groupby(df.player_id).cumsum()
        if W is None:
            S = cx.groupby(df.player_id).shift(1)
            N = cm.groupby(df.player_id).shift(1)
        else:
            S = cx.groupby(df.player_id).shift(1) - cx.groupby(df.player_id).shift(W + 1)
            N = cm.groupby(df.player_id).shift(1) - cm.groupby(df.player_id).shift(W + 1)
            # fewer than W earlier matches: fall back to the full history
            S = S.where(cx.groupby(df.player_id).shift(W + 1).notna(), cx.groupby(df.player_id).shift(1))
            N = N.where(cm.groupby(df.player_id).shift(W + 1).notna(), cm.groupby(df.player_id).shift(1))
        S = S.fillna(0)
        N = N.fillna(0)
        out[f'{tag}_{c}_{W or "all"}'] = (S + K * prior[c]) / (N + K)
        out[f'{n_name}_{W or "all"}'] = N
    return pd.DataFrame(out)


batted = df.batted
bowled = (df.balls_bowled > 0)
parts = []
for W in (5, 15, None):
    parts.append(roll(batted, bat_cols, W, 'bat', prior_bat, 'n_bat'))
    parts.append(roll(bowled, bowl_cols, W, 'bowl', prior_bowl, 'n_bowl'))
F = pd.concat(parts, axis=1)

# strike rate and economy from the shrunk numbers
for W in ('5', '15', 'all'):
    F[f'bat_sr_{W}'] = F[f'bat_runs_{W}'] / F[f'bat_balls_faced_{W}'].clip(lower=1) * 100
    F[f'bowl_econ_{W}'] = F[f'bowl_runs_conceded_{W}'] / (F[f'bowl_balls_bowled_{W}'].clip(lower=1) / 6)

# usage: share of the last 15 matches in which he batted / bowled
one = pd.Series(1.0, index=df.index)


def share(mask, W):
    x = mask.astype(float)
    cx = x.groupby(df.player_id).cumsum()
    c1 = one.groupby(df.player_id).cumsum()
    S = cx.groupby(df.player_id).shift(1) - cx.groupby(df.player_id).shift(W + 1).fillna(0)
    N = c1.groupby(df.player_id).shift(1) - c1.groupby(df.player_id).shift(W + 1).fillna(0)
    return (S.fillna(0) / N.replace(0, np.nan)).fillna(0)


F['p_bat_15'] = share(batted, 15)
F['p_bowl_15'] = share(bowled, 15)

# fielding rates (career, shrunk towards zero)
for c in fld_cols:
    x = df[c]
    cx = x.groupby(df.player_id).cumsum()
    c1 = one.groupby(df.player_id).cumsum()
    F[f'fld_{c}_rate'] = cx.groupby(df.player_id).shift(1).fillna(0) / (c1.groupby(df.player_id).shift(1).fillna(0) + K)

F['matches_prev'] = g.cumcount()
F['days_since'] = g['date'].diff().dt.days.fillna(365).clip(upper=365)

# ---- venue and team context (past matches only) ----
inn = (df.drop_duplicates(['match_id', 'team'])[['match_id', 'date', 'venue', 'team', 'opp', 'team_total']]
         .dropna(subset=['team_total']).sort_values(['date', 'match_id']))
gl = inn[inn.date < TRAIN_END].team_total.mean()     # global mean total


def past_mean(keys, val, K2=5, prior=gl):
    grp = inn.groupby(keys)[val]
    S = grp.cumsum() - inn[val]      # sum of earlier matches only
    N = grp.cumcount()
    return (S + K2 * prior) / (N + K2)


inn['venue_avg_total'] = past_mean('venue', 'team_total')


def team_roll(col_by, col_val, name, W=20):
    r = inn.groupby(col_by)[col_val].transform(lambda s: s.shift(1).rolling(W, min_periods=3).mean())
    inn[name] = r.fillna(gl)


# runs the team conceded = the opponent's total in the same match
inn['runs_against'] = inn.merge(
    inn[['match_id', 'team', 'team_total']].rename(columns={'team': 'opp', 'team_total': 'opp_t'}),
    on=['match_id', 'opp'], how='left')['opp_t'].values
team_roll('team', 'team_total', 'team_bat_form')
team_roll('team', 'runs_against', 'team_conceded_form')

opp_form = inn[['match_id', 'team', 'team_conceded_form', 'team_bat_form']].rename(
    columns={'team': 'opp', 'team_conceded_form': 'opp_conceded_form', 'team_bat_form': 'opp_bat_form'})
ctx = inn[['match_id', 'team', 'venue_avg_total', 'team_bat_form', 'team_conceded_form']]

df2 = pd.concat([df, F], axis=1)
df2 = df2.merge(ctx, on=['match_id', 'team'], how='left').merge(opp_form, on=['match_id', 'opp'], how='left')
for c in ['venue_avg_total', 'team_bat_form', 'team_conceded_form', 'opp_conceded_form', 'opp_bat_form']:
    df2[c] = df2[c].fillna(gl)

# two extra columns used later by the selector
df2['prev_date'] = df2.groupby('player_id')['date'].shift(1)
df2['stump_prev'] = df2.groupby('player_id')['stumpings'].cumsum() - df2['stumpings']

df2.to_pickle('feat_all.pkl')
print(df2.shape, (df2.match_id == 'PHANTOM').sum(), 'phantom rows')

(113297, 119) 5166 phantom rows


In [11]:
import time, pickle, warnings
from itertools import product
import numpy as np, pandas as pd
import xgboost as xgb
from scipy.stats import spearmanr
from sklearn.linear_model import Ridge
from sklearn.neural_network import MLPRegressor
from sklearn.preprocessing import StandardScaler
from sklearn.pipeline import make_pipeline
from sklearn.metrics import mean_absolute_error, mean_squared_error

warnings.filterwarnings('ignore')
rng = np.random.default_rng(42)

VAL_START, TEST_START = pd.Timestamp('2024-01-01'), pd.Timestamp('2025-01-01')
MIN_APP = 15        # same filter as the paper: players with >= 15 T20 appearances
N_SEEDS = 5
N_BOOT = 400
t0 = time.time()
tick = lambda msg: print(f'[{time.time() - t0:6.0f}s] {msg}')

f = pd.read_pickle('feat_all.pkl')
real = f[f.match_id != 'PHANTOM'].copy()
real = real[real.groupby('player_id').player_id.transform('size') >= MIN_APP]

# ---- features: no toss / innings (selection happens before the toss), team_type stays out ----
common = ['is_intl', 'days_since', 'matches_prev', 'venue_avg_total', 'p_bat_15', 'p_bowl_15']
win = ('5', '15', 'all')
bat_f = common + ['team_bat_form', 'opp_conceded_form'] \
    + [c for c in f.columns if c.startswith('bat_') and c.split('_')[-1] in win] \
    + [c for c in f.columns if c.startswith('n_bat_')]
bowl_f = common + ['team_conceded_form', 'opp_bat_form'] \
    + [c for c in f.columns if c.startswith('bowl_') and c.split('_')[-1] in win] \
    + [c for c in f.columns if c.startswith('n_bowl_')]
bat_f, bowl_f = list(dict.fromkeys(bat_f)), list(dict.fromkeys(bowl_f))

RAW = {'runs', 'balls_faced', 'fours', 'sixes', 'dot_faced', 'is_out', 'wickets', 'balls_bowled',
       'runs_conceded', 'dot_bowled', 'wides', 'noballs', 'catches', 'stumpings', 'run_outs', 'batted'}
for fl in (bat_f, bowl_f):
    assert not RAW & set(fl), RAW & set(fl)          # no same-match stat may be a feature
    assert real[fl].notna().all().all(), real[fl].isna().sum()[lambda s: s > 0]
print(f'features: bat {len(bat_f)} | bowl {len(bowl_f)} | rows {len(real):,} | players {real.player_id.nunique():,}')

TASKS = {
    'bat':  dict(mask=real.batted == 1,        feats=bat_f,  target='runs',    base='bat_runs_15'),
    'bowl': dict(mask=real.balls_bowled > 0,   feats=bowl_f, target='wickets', base='bowl_wickets_15'),
}
GRID = list(product((2, 3, 4), (0.03, 0.08), (10, 30)))     # depth, learning rate, min_child_weight


def xgb_model(depth, lr, mcw, n, seed, **kw):
    return xgb.XGBRegressor(objective='count:poisson', n_estimators=n, learning_rate=lr, max_depth=depth,
                            min_child_weight=mcw, subsample=0.8, colsample_bytree=0.7, reg_lambda=5,
                            tree_method='hist', random_state=seed, n_jobs=-1, **kw)


def tune_xgb(tr, va, feats, target):
    """grid search on the 2024 validation year, early stopping on Poisson deviance"""
    best = None
    for depth, lr, mcw in GRID:
        m = xgb_model(depth, lr, mcw, 3000, 0, early_stopping_rounds=50)
        m.fit(tr[feats], tr[target], eval_set=[(va[feats], va[target])], verbose=False)
        if best is None or m.best_score < best['score']:
            best = dict(score=m.best_score, depth=depth, lr=lr, mcw=mcw, n=m.best_iteration + 1)
    return best


def cluster_boot(te, err, n_boot=N_BOOT):
    """match-level bootstrap: resample matches, not rows (players of one match are not independent).
    Returns dict name -> array of bootstrap MAEs, so any difference can be taken afterwards."""
    codes, uniq = pd.factorize(te.match_id.values)
    cnt = np.bincount(codes)
    W = rng.multinomial(len(uniq), np.full(len(uniq), 1 / len(uniq)), size=n_boot)   # (B, matches)
    den = W @ cnt
    return {k: (W @ np.bincount(codes, weights=v)) / den for k, v in err.items()}


ci = lambda x: np.percentile(x, [2.5, 97.5])
saved, test_out = {}, {}

for task, T in TASKS.items():
    feats, target = T['feats'], T['target']
    d = real[T['mask']]
    tr, va = d[d.date < VAL_START], d[(d.date >= VAL_START) & (d.date < TEST_START)]
    trva, te = d[d.date < TEST_START], d[d.date >= TEST_START]
    print(f'\n===== {task}: train {len(tr):,} | val {len(va):,} | test {len(te):,} ({te.match_id.nunique()} matches)')

    # ---- XGBoost (Poisson): tune on val, refit on train+val with ~10% more trees ----
    hp = tune_xgb(tr, va, feats, target)
    n_final = int(hp['n'] * 1.1)
    tick(f"{task} xgb best: depth {hp['depth']}, lr {hp['lr']}, mcw {hp['mcw']}, trees {hp['n']} -> final {n_final}")
    xg = [xgb_model(hp['depth'], hp['lr'], hp['mcw'], n_final, s).fit(trva[feats], trva[target]) for s in range(N_SEEDS)]
    p_xgb = lambda X: np.mean([m.predict(X[feats]) for m in xg], 0)

    # ---- Ridge ----
    rd = make_pipeline(StandardScaler(), Ridge(alpha=50)).fit(trva[feats], trva[target])
    p_ridge = lambda X: np.clip(rd.predict(X[feats]), 0, None)

    # ---- ANN ----
    ann = [make_pipeline(StandardScaler(), MLPRegressor(hidden_layer_sizes=(64, 32), alpha=0.1, early_stopping=True,
           validation_fraction=0.15, n_iter_no_change=15, max_iter=400, random_state=s)
           ).fit(trva[feats], trva[target]) for s in range(N_SEEDS)]
    p_ann = lambda X: np.clip(np.mean([m.predict(X[feats]) for m in ann], 0), 0, None)
    tick(f'{task} ridge + ann done')

    # ---- test predictions ----
    P = {'baseline (rolling 15)': te[T['base']].values,
         'train mean': np.full(len(te), trva[target].mean()),
         'Ridge': p_ridge(te), 'XGBoost': p_xgb(te), 'ANN': p_ann(te)}
    y = te[target].values
    err = {k: np.abs(y - v) for k, v in P.items()}
    B = cluster_boot(te, err)

    rows = []
    for k, v in P.items():
        lo, hi = ci(B[k])
        rows.append(dict(model=k, MAE=err[k].mean(), MAE_lo=lo, MAE_hi=hi,
                         RMSE=np.sqrt(mean_squared_error(y, v)), Spearman=spearmanr(y, v)[0]))
    print(pd.DataFrame(rows).set_index('model').round(3).to_string())

    print('MAE difference, 95% match-level bootstrap CI (negative = first model better)')
    for a, b in [('XGBoost', 'baseline (rolling 15)'), ('ANN', 'baseline (rolling 15)'), ('Ridge', 'baseline (rolling 15)'),
                 ('XGBoost', 'Ridge'), ('ANN', 'XGBoost')]:
        dlt = B[a] - B[b]; lo, hi = ci(dlt)
        print(f'  {a:8s} - {b:22s} {err[a].mean() - err[b].mean():+.3f}  [{lo:+.3f}, {hi:+.3f}]')

    # ---- is_intl check: IPL rows are is_intl=0 in training, but selection always uses is_intl=1 ----
    ipl = te[te.src == 'ipl']
    flip = ipl.assign(is_intl=1)
    print(f'is_intl check: train rows intl {int(trva.is_intl.sum()):,} / ipl {int((1 - trva.is_intl).sum()):,}')
    print(f'  test IPL rows {len(ipl):,}: mean XGB pred as-is {p_xgb(ipl).mean():.3f} | with is_intl=1 {p_xgb(flip).mean():.3f} '
          f'| actual mean {ipl[target].mean():.3f}')
    for s_, g_ in te.groupby('src'):
        i_ = te.index.get_indexer(g_.index)
        print(f'  src={s_:5s} n={len(g_):6,}  MAE baseline {err["baseline (rolling 15)"][i_].mean():.3f}'
              f' | XGB {err["XGBoost"][i_].mean():.3f}')

    saved[task] = dict(feats=feats, target=target, xgb=xg, ridge=rd, ann=ann, xgb_hp={**hp, 'n_final': n_final})
    test_out[task] = te[['match_id', 'player_id', 'date', 'src', target]].assign(
        **{k: v for k, v in P.items()})

pickle.dump(saved, open('models_step4.pkl', 'wb'))
pickle.dump(test_out, open('test_preds_step4.pkl', 'wb'))
tick('saved models_step4.pkl and test_preds_step4.pkl')

features: bat 41 | bowl 38 | rows 90,226 | players 1,891

===== bat: train 41,375 | val 9,267 | test 15,151 (1210 matches)
[    26s] bat xgb best: depth 4, lr 0.08, mcw 30, trees 138 -> final 151
[    73s] bat ridge + ann done
                          MAE  MAE_lo  MAE_hi    RMSE  Spearman
model                                                          
baseline (rolling 15)  14.218  13.982  14.449  19.375     0.347
train mean             15.370  15.151  15.593  20.553       NaN
Ridge                  13.684  13.442  13.920  18.961     0.389
XGBoost                13.683  13.437  13.930  18.969     0.390
ANN                    13.651  13.395  13.885  18.954     0.391
MAE difference, 95% match-level bootstrap CI (negative = first model better)
  XGBoost  - baseline (rolling 15)  -0.535  [-0.599, -0.471]
  ANN      - baseline (rolling 15)  -0.567  [-0.629, -0.504]
  Ridge    - baseline (rolling 15)  -0.534  [-0.600, -0.468]
  XGBoost  - Ridge                  -0.001  [-0.030, +0.026]
  AN

In [38]:
import pickle
import time
import numpy as np
import pandas as pd

TEST_START = pd.Timestamp('2025-01-01')
WINDOWS = (12,)            # look-back window in months
MIN_PREV = 5               # min T20 appearances before the match date
IPL_MIN_MATCHES = 8        # IPL entrants need this many batting (or bowling) innings in the window
K_BAT, K_BOWL = 8, 6       # IPL entrants admitted: top-K by runs per innings / wickets per innings

# Publicly announced international retirements, with the announcement date.
# A player is dropped from the pool from that date on (information a selector has).
RETIRED_T20I = {
    'V Kohli': '2024-06-29',
    'RG Sharma': '2024-06-29',
    'RA Jadeja': '2024-06-29',
    'R Ashwin': '2024-12-18',
}

f = pd.read_pickle('feat_all.pkl')
real = f[f.match_id != 'PHANTOM']
elig = pd.read_pickle('eligibility.pkl')
eligible = set(elig.index[elig.india_eligible])
names = dict(zip(real.player_id, real.player))

# retirement dates by player id
retired = {}
for n, d in RETIRED_T20I.items():
    for p in real.loc[real.player == n, 'player_id'].unique():
        retired[p] = pd.Timestamp(d)

# appearance dates per player (any T20), for the MIN_PREV filter
pdates = {p: np.sort(g.date.values) for p, g in real.groupby('player_id')}
n_before = lambda p, d: int(np.searchsorted(pdates[p], np.datetime64(d), 'left'))

# India T20I appearances and IPL rows, sorted by date for window slicing
ind_rows = real[(real.src == 'intl') & (real.team == 'India')].sort_values('date')
ind_date, ind_pid = ind_rows.date.values, ind_rows.player_id.values
ipl_rows = real[(real.src == 'ipl') & real.player_id.isin(eligible)].sort_values('date').reset_index(drop=True)
ipl_date = ipl_rows.date.values


def window_slice(dates, date, months):
    lo = date - pd.DateOffset(months=months)
    return (np.searchsorted(dates, np.datetime64(lo), 'left'),
            np.searchsorted(dates, np.datetime64(date), 'left'))     # match day itself is excluded


def build_pool(date, months=12, min_prev=MIN_PREV):
    # group A: played for India in the window
    i0, i1 = window_slice(ind_date, date, months)
    grp_a = {p for p in set(ind_pid[i0:i1]) if p in eligible}

    # group B: IPL players with no India T20I in the window, admitted by rule (past-only stats)
    j0, j1 = window_slice(ipl_date, date, months)
    w = ipl_rows.iloc[j0:j1]
    w = w[~w.player_id.isin(grp_a)]
    bat = w[w.batted == 1].groupby('player_id').runs.agg(['mean', 'size'])
    bat = bat[bat['size'] >= IPL_MIN_MATCHES].sort_values('mean', ascending=False).head(K_BAT)
    bw = w[w.balls_bowled > 0].groupby('player_id').wickets.agg(['mean', 'size'])
    bw = bw[bw['size'] >= IPL_MIN_MATCHES].sort_values('mean', ascending=False).head(K_BOWL)
    grp_b = set(bat.index) | set(bw.index)

    pool = []
    for p in grp_a | grp_b:
        if n_before(p, date) < min_prev:
            continue
        if p in retired and date >= retired[p]:
            continue
        pool.append(p)
    return sorted(pool), grp_a, grp_b


ind = real[(real.team == 'India') & (real.src == 'intl')]
mt = ind.groupby('match_id').agg(date=('date', 'first')).sort_values('date')
mt = mt[mt.date >= TEST_START]
xi = {m: set(g.player_id) for m, g in ind.groupby('match_id')}

pools, rows, misses = {}, [], []
t0 = time.time()
for m in WINDOWS:
    for mid, r in mt.iterrows():
        pool, a, b = build_pool(r.date, m)
        pools[(m, mid)] = pool
        miss = xi[mid] - set(pool)
        rows.append(dict(months=m, match_id=mid, date=r.date, pool=len(pool),
                         n_india=len(set(pool) & a), n_ipl_entrants=len(set(pool) & b - a),
                         covered=11 - len(miss)))
        for p in miss:
            misses.append(dict(match_id=mid, date=r.date, player=names[p],
                               why='not_eligible' if p not in eligible else
                               'retired' if (p in retired and r.date >= retired[p]) else
                               'not_in_rule'))

R = pd.DataFrame(rows)
print(f'built {len(R)} pools in {time.time() - t0:.1f}s')
print(R.groupby('months').agg(pool_mean=('pool', 'mean'), pool_min=('pool', 'min'), pool_max=('pool', 'max'),
                              india_mean=('n_india', 'mean'), ipl_entrants_mean=('n_ipl_entrants', 'mean'),
                              covered_mean=('covered', 'mean')).round(1).to_string())
if misses:
    M_ = pd.DataFrame(misses)
    print('\nreal XI players missing from the pool:', len(M_), 'player-matches')
    print(M_.why.value_counts().to_string())
    print(M_.groupby('player').size().sort_values(ascending=False).head(8).to_string())

pickle.dump(dict(pools=pools, matches=mt, xi=xi, min_prev=MIN_PREV), open('pools_step5.pkl', 'wb'))
print('\nsaved pools_step5.pkl')

built 55 pools in 0.6s
        pool_mean  pool_min  pool_max  india_mean  ipl_entrants_mean  covered_mean
months                                                                            
12           36.1        31        45        23.3               12.8          10.8

real XI players missing from the pool: 11 player-matches
why
not_in_rule    11
player
Ashok Sharma      1
Ishan Kishan      1
JM Sharma         1
Kuldeep Yadav     1
MP Yadav          1
Mohammed Shami    1
Mohammed Siraj    1
Naman Dhir        1

saved pools_step5.pkl


In [39]:
import pickle
import numpy as np
import pandas as pd

WINDOWS = (12,)
P_BOWL_MIN, STUMP_MIN = 0.35, 2

# blend weights kept from the earlier version; they are NOT tuned yet (see note below)
W_INTL, W_IPL = 0.80, 0.20      # players who already have international experience
IPL_ONLY_SCALE = 0.40           # players with no international match before the date
MIN_INTL_EXP = 1                # international matches needed before the match date

# Load data, models, and Step 5 pool
f = pd.read_pickle('feat_all.pkl')
real = f[f.match_id != 'PHANTOM']
M = pickle.load(open('models_step4.pkl', 'rb'))
P5 = pickle.load(open('pools_step5.pkl', 'rb'))
pools, mt, xi = P5['pools'], P5['matches'], P5['xi']

bat_f, bowl_f = M['bat']['feats'], M['bowl']['feats']
pred = lambda ms, X: np.mean([m.predict(X) for m in ms], 0)
names = dict(zip(real.player_id, real.player))

fs = f.sort_values(['player_id', 'date']).reset_index(drop=True)
pdates = {p: g.date.values for p, g in fs.groupby('player_id')}
first = fs.groupby('player_id').head(1)
base_idx = dict(zip(first.player_id, first.index))
asof_row = (
    lambda p, d: base_idx[p]
    + np.searchsorted(pdates[p], np.datetime64(d), 'left')
)

# international appearance dates per player: used to decide "has international
# experience" from matches strictly before the target date
intl_dates = {p: np.sort(g.date.values)
              for p, g in real[real.src == 'intl'].groupby('player_id')}


def n_intl_before(p, d):
  if p not in intl_dates:
    return 0
  return int(np.searchsorted(intl_dates[p], np.datetime64(d), 'left'))


CTX = [
    'venue_avg_total',
    'team_bat_form',
    'team_conceded_form',
    'opp_conceded_form',
    'opp_bat_form',
]
ind = real[(real.team == 'India') & (real.src == 'intl')]
ctx_of = ind.groupby('match_id')[CTX].first()

flag_changed, flag_total = [], []     # diagnostic: how often the old flag disagreed


def score_pool_weighted(pool, date, ctx):
  X = fs.loc[[asof_row(p, date) for p in pool]].reset_index(drop=True)
  assert (X.player_id.values == np.array(pool)).all()
  assert (X.prev_date.isna() | (X.prev_date < date)).all(), 'as-of row sees the future'

  X['days_since'] = (date - X.prev_date).dt.days.fillna(365).clip(upper=365)
  for k, v in ctx.items():
    X[k] = v

  # 1. Prediction under T20I International Match Context
  X_intl = X.copy()
  X_intl['is_intl'] = 1
  er_intl = pred(M['bat']['xgb'], X_intl[bat_f])
  ew_intl = pred(M['bowl']['xgb'], X_intl[bowl_f])

  # 2. Prediction under IPL Match Context
  X_ipl = X.copy()
  X_ipl['is_intl'] = 0
  er_ipl = pred(M['bat']['xgb'], X_ipl[bat_f])
  ew_ipl = pred(M['bowl']['xgb'], X_ipl[bowl_f])

  # experience is judged from the past only (not from the player's next match)
  n_intl = np.array([n_intl_before(p, date) for p in pool])
  has_intl_exp = n_intl >= MIN_INTL_EXP

  old_flag = ((X.matches_prev > 0) & (X.is_intl == 1)).values
  flag_changed.append(int((old_flag != has_intl_exp).sum()))
  flag_total.append(len(pool))

  er = np.where(has_intl_exp, W_INTL * er_intl + W_IPL * er_ipl, IPL_ONLY_SCALE * er_ipl)
  ew = np.where(has_intl_exp, W_INTL * ew_intl + W_IPL * ew_ipl, IPL_ONLY_SCALE * ew_ipl)

  p_bat = X.p_bat_15.values
  p_bowl = X.p_bowl_15.values

  is_bowler = p_bowl >= P_BOWL_MIN
  is_pure_bat = (p_bat >= 0.70) & (~is_bowler)
  is_pure_bowl = is_bowler & (p_bat < 0.40)

  return pd.DataFrame(
      dict(
          player_id=pool,
          matches_prev=X.matches_prev.values,
          p_bat=p_bat,
          p_bowl=p_bowl,
          e_runs=er,
          e_wkts=ew,
          bat=p_bat * er,
          bowl=p_bowl * ew,
          bat_form=p_bat * X.bat_runs_15.values,
          bowl_form=p_bowl * X.bowl_wickets_15.values,
          keeper=(X.stump_prev.values >= STUMP_MIN),
          bowler=is_bowler,
          pure_bat=is_pure_bat,
          pure_bowl=is_pure_bowl,
      )
  )


parts = []
for m in WINDOWS:
  for mid, r in mt.iterrows():
    S = score_pool_weighted(pools[(m, mid)], r.date, ctx_of.loc[mid].to_dict())
    S.insert(0, 'match_id', mid)
    S.insert(0, 'months', m)
    S.insert(2, 'date', r.date)
    parts.append(S)

scores = pd.concat(parts, ignore_index=True)

print(f'old flag disagreed with the past-only flag for {sum(flag_changed)} of '
      f'{sum(flag_total)} scored players ({100 * sum(flag_changed) / sum(flag_total):.1f}%)')

# Latest match top candidates preview
last = scores[scores.match_id == mt.index[-1]].copy()
last['name'] = last.player_id.map(names)
cols = ['name', 'bat', 'bowl', 'keeper', 'bowler']

print('\nTop 10 Batting Candidates:')
print(last.sort_values('bat', ascending=False)[cols].head(10).to_string(index=False))
print('\nTop 10 Bowling Candidates:')
print(last.sort_values('bowl', ascending=False)[cols].head(10).to_string(index=False))

scores.to_pickle('scores_step6.pkl')
print('\nSaved updated scores_step6.pkl', scores.shape)

old flag disagreed with the past-only flag for 746 of 1983 scored players (37.6%)

Top 10 Batting Candidates:
           name       bat     bowl  keeper  bowler
       KL Rahul 33.335800 0.000000    True   False
B Sai Sudharsan 33.199032 0.000000   False   False
   Shubman Gill 32.642357 0.000000   False   False
  V Suryavanshi 30.681168 0.000000   False   False
      SV Samson 29.559734 0.000000    True   False
       SA Yadav 28.560093 0.000000   False   False
   Ishan Kishan 28.351730 0.000000    True   False
Abhishek Sharma 28.092634 0.115206   False   False
        SS Iyer 26.758032 0.000000   False   False
    Dhruv Jurel 26.045315 0.000000    True   False

Top 10 Bowling Candidates:
             name      bat     bowl  keeper  bowler
        JJ Bumrah 0.378574 1.306075   False    True
M Prasidh Krishna 0.598335 1.220045   False    True
      Yash Thakur 0.000000 1.219329   False    True
   Mohammed Siraj 0.435023 1.203207   False    True
         MP Yadav 0.698221 1.162309   Fal

In [42]:
import pickle
import time
import numpy as np
import pandas as pd
from scipy.optimize import milp, LinearConstraint, Bounds
from pymoo.algorithms.moo.nsga2 import NSGA2
from pymoo.core.crossover import Crossover
from pymoo.core.mutation import Mutation
from pymoo.core.problem import Problem
from pymoo.core.repair import Repair
from pymoo.core.sampling import Sampling
from pymoo.optimize import minimize

SIZE = 11
MIN_BOWLERS = 5
MIN_PURE_BAT = 3
MIN_PURE_BOWL = 2

POP, GEN, SEED = 100, 200, 1
MATCH_LIMIT = None         # dry run first; set to None to run all matches

scores = pd.read_pickle('scores_step6.pkl')
P5 = pickle.load(open('pools_step5.pkl', 'rb'))
xi = P5['xi']
rng = np.random.default_rng(SEED)


class XIProblemRoleConstrained(Problem):

  def __init__(self, bat, bowl, keeper, bowler, pure_bat, pure_bowl):
    super().__init__(
        n_var=len(bat), n_obj=2, n_ieq_constr=4, xl=0, xu=1, vtype=bool
    )
    self.bat, self.bowl = bat, bowl
    self.keeper = keeper.astype(float)
    self.bowler = bowler.astype(float)
    self.pure_bat = pure_bat.astype(float)
    self.pure_bowl = pure_bowl.astype(float)

  def _evaluate(self, X, out, *args, **kwargs):
    X = X.astype(float)
    out['F'] = np.column_stack([-(X @ self.bat), -(X @ self.bowl)])
    out['G'] = np.column_stack([
        1 - X @ self.keeper,
        MIN_BOWLERS - X @ self.bowler,
        MIN_PURE_BAT - X @ self.pure_bat,
        MIN_PURE_BOWL - X @ self.pure_bowl,
    ])


class RoleSampling(Sampling):

  def _do(self, problem, n_samples, **kwargs):
    X = np.zeros((n_samples, problem.n_var), dtype=bool)
    for i in range(n_samples):
      trials = 0
      while True:
        trials += 1
        idx = rng.choice(problem.n_var, SIZE, replace=False)
        cond = (
            problem.keeper[idx].sum() >= 1
            and problem.bowler[idx].sum() >= MIN_BOWLERS
            and problem.pure_bat[idx].sum() >= MIN_PURE_BAT
            and problem.pure_bowl[idx].sum() >= MIN_PURE_BOWL
        )
        if cond or trials > 1000:
          break
      X[i, idx] = True
    return X


class UnionCrossover(Crossover):

  def __init__(self, prob=0.9):
    super().__init__(2, 2, prob=prob)

  def _do(self, problem, X, **kwargs):
    X = np.asarray(X, dtype=bool)
    Y = X.copy()
    for i in range(X.shape[1]):
      a, b = X[0, i], X[1, i]
      both, either = np.flatnonzero(a & b), np.flatnonzero(a ^ b)
      need = SIZE - len(both)
      for j in range(2):
        Y[j, i] = False
        Y[j, i, both] = True
        if need > 0:
          Y[j, i, rng.choice(either, need, replace=False)] = True
    return Y


class FixSize(Repair):

  def _do(self, problem, X, **kwargs):
    X = np.asarray(X, dtype=bool).copy()
    for i in range(len(X)):
      on, off = np.flatnonzero(X[i]), np.flatnonzero(~X[i])
      k = len(on) - SIZE
      if k > 0:
        X[i, rng.choice(on, k, replace=False)] = False
      elif k < 0:
        X[i, rng.choice(off, -k, replace=False)] = True
    return X


class SwapMutation(Mutation):

  def _do(self, problem, X, **kwargs):
    X = np.asarray(X, dtype=bool).copy()
    for i in range(len(X)):
      on, off = np.flatnonzero(X[i]), np.flatnonzero(~X[i])
      k = rng.integers(1, 3)
      X[i, rng.choice(on, k, replace=False)] = False
      X[i, rng.choice(off, k, replace=False)] = True
    return X


def knee(F):
  if len(F) < 3:
    return int(np.argmax(F.sum(1)))
  rngs = np.where(np.ptp(F, 0) == 0, 1, np.ptp(F, 0))
  Fn = (F - F.min(0)) / rngs
  a, b = Fn[np.argmax(Fn[:, 0])], Fn[np.argmax(Fn[:, 1])]
  d = b - a
  if np.allclose(d, 0):
    return int(np.argmax(Fn.sum(1)))
  dist = np.abs(d[0] * (Fn[:, 1] - a[1]) - d[1] * (Fn[:, 0] - a[0])) / np.hypot(*d)
  return int(np.argmax(dist))


def exact_max(c, keeper, bowler, pure_bat, pure_bowl):
  """exact best value of one objective under the same five constraints (MILP)"""
  n = len(c)
  A = np.vstack([np.ones(n), keeper.astype(float), bowler.astype(float),
                 pure_bat.astype(float), pure_bowl.astype(float)])
  lb = [SIZE, 1, MIN_BOWLERS, MIN_PURE_BAT, MIN_PURE_BOWL]
  ub = [SIZE, np.inf, np.inf, np.inf, np.inf]
  r = milp(-np.asarray(c, float), constraints=LinearConstraint(A, lb, ub),
           integrality=np.ones(n), bounds=Bounds(0, 1))
  return -r.fun if r.status == 0 else np.nan


results, rows = {}, []
t0 = time.time()
groups = list(scores.groupby(['months', 'match_id'], sort=False))
if MATCH_LIMIT:
  groups = groups[:MATCH_LIMIT]
total = len(groups)

print(f'STEP 7: NSGA-II on {total} problems (pop {POP}, gen {GEN})\n')

for i, ((m, mid), S) in enumerate(groups, 1):
  S = S.reset_index(drop=True)

  # if the pool has no flagged keeper, fall back to likely batters (and log it)
  keeper_arr = S.keeper.values
  used_fallback = keeper_arr.sum() == 0
  if used_fallback:
    keeper_arr = S.p_bat.values >= 0.5

  prob = XIProblemRoleConstrained(
      S.bat.values, S.bowl.values, keeper_arr,
      S.bowler.values, S.pure_bat.values, S.pure_bowl.values,
  )
  algo = NSGA2(
      pop_size=POP, sampling=RoleSampling(), crossover=UnionCrossover(prob=0.9),
      mutation=SwapMutation(), repair=FixSize(), eliminate_duplicates=True,
  )
  res = minimize(prob, algo, ('n_gen', GEN), seed=SEED, verbose=False)

  ok = np.asarray(res.opt.get('feasible')).ravel()
  feasible_found = bool(ok.any())
  if feasible_found:
    F, X = -res.opt.get('F')[ok], res.opt.get('X')[ok]
  else:                                  # no feasible XI: keep the result but flag it
    F, X = -res.opt.get('F'), res.opt.get('X')

  k = knee(F)
  pid = S.player_id.values
  knee_xi = set(pid[X[k]])
  results[(m, mid)] = dict(F=F, X=[set(pid[x]) for x in X], knee=k, knee_xi=knee_xi,
                           feasible=feasible_found)

  # how far the two ends of the front are from the exact optimum
  gap_bat = gap_bowl = np.nan
  if feasible_found:
    args = (keeper_arr, S.bowler.values, S.pure_bat.values, S.pure_bowl.values)
    e1, e2 = exact_max(S.bat.values, *args), exact_max(S.bowl.values, *args)
    gap_bat = 100 * (e1 - F[:, 0].max()) / e1
    gap_bowl = 100 * (e2 - F[:, 1].max()) / e2

  # does the real XI itself satisfy the role rules? (players of the XI that are in the pool)
  a = S[S.player_id.isin(xi[mid])]
  real_ok = bool(a.keeper.sum() >= 1 and a.bowler.sum() >= MIN_BOWLERS
                 and a.pure_bat.sum() >= MIN_PURE_BAT and a.pure_bowl.sum() >= MIN_PURE_BOWL)

  rows.append(dict(
      months=m, match_id=mid, pool=len(S), front=len(F),
      overlap=len(knee_xi & xi[mid]),
      rand_exp=11 * len(a) / len(S),          # expected overlap of a random 11 from this pool
      feasible_found=feasible_found, keeper_fallback=used_fallback, real_xi_feasible=real_ok,
      gap_bat=gap_bat, gap_bowl=gap_bowl))

  if i % 5 == 0 or i == total:
    el = time.time() - t0
    print(f'[{i:2d}/{total}] Elapsed {el / 60:.1f}m | ETA {el / i * (total - i) / 60:.1f}m | '
          f'pool {len(S)} | overlap {rows[-1]["overlap"]}/11')

R = pd.DataFrame(rows)
print(f'\nFinished in {time.time() - t0:.1f}s\n')
print(R.groupby('months').agg(
    pool=('pool', 'mean'), front=('front', 'mean'),
    knee_overlap=('overlap', 'mean'), random_expected=('rand_exp', 'mean'),
    feasible_found=('feasible_found', 'mean'), keeper_fallback=('keeper_fallback', 'sum'),
    real_xi_feasible=('real_xi_feasible', 'mean'),
    gap_bat_max=('gap_bat', 'max'), gap_bowl_max=('gap_bowl', 'max')).round(3).T.to_string())

pickle.dump(dict(results=results, summary=R), open('nsga_step7.pkl', 'wb'))
print('\nSaved nsga_step7.pkl')

STEP 7: NSGA-II on 55 problems (pop 100, gen 200)

[ 5/55] Elapsed 0.5m | ETA 5.1m | pool 45 | overlap 3/11
[10/55] Elapsed 1.0m | ETA 4.6m | pool 36 | overlap 4/11
[15/55] Elapsed 1.5m | ETA 4.1m | pool 34 | overlap 2/11
[20/55] Elapsed 2.1m | ETA 3.6m | pool 33 | overlap 4/11
[25/55] Elapsed 2.6m | ETA 3.1m | pool 33 | overlap 4/11
[30/55] Elapsed 3.1m | ETA 2.6m | pool 32 | overlap 4/11
[35/55] Elapsed 3.6m | ETA 2.1m | pool 32 | overlap 5/11
[40/55] Elapsed 4.1m | ETA 1.5m | pool 36 | overlap 3/11
[45/55] Elapsed 4.6m | ETA 1.0m | pool 39 | overlap 2/11
[50/55] Elapsed 5.1m | ETA 0.5m | pool 40 | overlap 2/11
[55/55] Elapsed 5.6m | ETA 0.0m | pool 42 | overlap 1/11

Finished in 336.0s

months                12
pool              36.055
front             95.418
knee_overlap       2.909
random_expected    3.334
feasible_found     1.000
keeper_fallback    0.000
real_xi_feasible   0.855
gap_bat_max        0.795
gap_bowl_max       2.431

Saved nsga_step7.pkl


In [35]:

import pickle
import numpy as np
import pandas as pd
from pymoo.algorithms.moo.nsga2 import NSGA2
from pymoo.core.crossover import Crossover
from pymoo.core.mutation import Mutation
from pymoo.core.problem import Problem
from pymoo.core.repair import Repair
from pymoo.core.sampling import Sampling
from pymoo.optimize import minimize

# 1. Load Trained XGBoost Models and Historical Dataset
f = pd.read_pickle('feat_all.pkl')
real = f[f.match_id != 'PHANTOM']
M = pickle.load(open('models_step4.pkl', 'rb'))
elig = pd.read_pickle('eligibility.pkl')
eligible_ids = set(elig.index[elig.india_eligible])
names = dict(zip(real.player_id, real.player))

# 2. Upcoming Match Context Parameters (Dec 2026 IND vs SL T20I)
MATCH_DATE = pd.Timestamp('2026-12-15')
VENUE_AVG_TOTAL = 172.0  # Projected T20 Venue Avg
OPP_CONCEDED_FORM = 8.4  # Sri Lanka T20 Bowling Form Metric
TEAM_BAT_FORM = 8.8

# 3. Explicit 15 IPL Youngsters & Hard Exclusion Lists
IPL_YOUNGSTERS = {
    'Vaibhav Sooryavanshi',
    'V Suryavanshi',
    'Priyansh Arya',
    'Ayush Mhatre',
    'Kartik Sharma',
    'Sameer Rizvi',
    'Angkrish Raghuvanshi',
    'Nehal Wadhera',
    'Auqib Nabi',
    'Prashant Veer',
    'Harsh Dubey',
    'Prince Yadav',
    'MP Yadav',
    'Vaibhav Arora',
    'Vipraj Nigam',
    'Ashwani Kumar',
    'Digvesh Rathi',
    'DS Rathi',
}

EXCLUDE_PLAYERS = {
    'V Kohli',
    'RG Sharma',
    'RA Jadeja',
    'R Ashwin',
    'AM Rahane',
    'Avesh Khan',
    'Mohammed Shami',
    'Mohammed Siraj',
    'KL Rahul',
    'RP Pant',
    'R Gaikwad',
    'RD Gaikwad',
    'YS Chahal',
    'RM Patidar',
    'TU Deshpande',
}

exclude_ids = set(real[real.player.isin(EXCLUDE_PLAYERS)].player_id.unique())
ipl_allowed_ids = set(
    real[real.player.isin(IPL_YOUNGSTERS)].player_id.unique()
)

# 4. Filter Candidate Pool (Active India T20I Core + IPL Youngsters List)
latest_records = (
    real.sort_values('date').groupby('player_id').last().reset_index()
)
candidate_pool = []

for idx, r in latest_records.iterrows():
  p = r.player_id
  if p not in eligible_ids or p in exclude_ids:
    continue

  p_ind = real[
      (real.player_id == p)
      & (real.team == 'India')
      & (real.src == 'intl')
      & (real.date <= MATCH_DATE)
  ]
  if len(p_ind) > 0 and (MATCH_DATE - p_ind.date.max()).days <= 365:
    candidate_pool.append(p)
    continue

  if p in ipl_allowed_ids:
    candidate_pool.append(p)

pool_df = latest_records[
    latest_records.player_id.isin(candidate_pool)
].copy()
pool = pool_df.player_id.tolist()

# 5. Calculate 80% T20I / 20% IPL Weighted Predictions
bat_f, bowl_f = M['bat']['feats'], M['bowl']['feats']
pred = lambda ms, X: np.mean([m.predict(X) for m in ms], 0)

pool_df['venue_avg_total'] = VENUE_AVG_TOTAL
pool_df['opp_conceded_form'] = OPP_CONCEDED_FORM
pool_df['team_bat_form'] = TEAM_BAT_FORM

# Intl Context Scoring
pool_df_intl = pool_df.copy()
pool_df_intl['is_intl'] = 1
er_intl = pred(M['bat']['xgb'], pool_df_intl[bat_f])
ew_intl = pred(M['bowl']['xgb'], pool_df_intl[bowl_f])

# IPL Context Scoring
pool_df_ipl = pool_df.copy()
pool_df_ipl['is_intl'] = 0
er_ipl = pred(M['bat']['xgb'], pool_df_ipl[bat_f])
ew_ipl = pred(M['bowl']['xgb'], pool_df_ipl[bowl_f])

has_intl_exp = (pool_df.matches_prev > 0) & (pool_df.is_intl == 1)
er = np.where(has_intl_exp, 0.80 * er_intl + 0.20 * er_ipl, 0.40 * er_ipl)
ew = np.where(has_intl_exp, 0.80 * ew_intl + 0.20 * ew_ipl, 0.40 * ew_ipl)

p_bat, p_bowl = pool_df.p_bat_15.values, pool_df.p_bowl_15.values
is_bowler = p_bowl >= 0.35

S = pd.DataFrame(
    dict(
        player_id=pool,
        bat=p_bat * er,
        bowl=p_bowl * ew,
        keeper=(pool_df.stump_prev.values >= 2),
        bowler=is_bowler,
        pure_bat=(p_bat >= 0.70) & (~is_bowler),
        pure_bowl=is_bowler & (p_bat < 0.40),
    )
)

# 6. Role-Constrained NSGA-II Optimization
SIZE, MIN_BOWLERS, MIN_PURE_BAT, MIN_PURE_BOWL = 11, 5, 4, 3
rng = np.random.default_rng(1)


class XIProblem(Problem):

  def __init__(self, bat, bowl, keeper, bowler, pure_bat, pure_bowl):
    super().__init__(
        n_var=len(bat), n_obj=2, n_ieq_constr=4, xl=0, xu=1, vtype=bool
    )
    self.bat, self.bowl = bat, bowl
    self.keeper = keeper.astype(float)
    self.bowler = bowler.astype(float)
    self.pure_bat = pure_bat.astype(float)
    self.pure_bowl = pure_bowl.astype(float)

  def _evaluate(self, X, out, *args, **kwargs):
    X = X.astype(float)
    out['F'] = np.column_stack([-(X @ self.bat), -(X @ self.bowl)])
    out['G'] = np.column_stack([
        1 - X @ self.keeper,
        MIN_BOWLERS - X @ self.bowler,
        MIN_PURE_BAT - X @ self.pure_bat,
        MIN_PURE_BOWL - X @ self.pure_bowl,
    ])


class RoleSampling(Sampling):

  def _do(self, problem, n_samples, **kwargs):
    X = np.zeros((n_samples, problem.n_var), dtype=bool)
    for i in range(n_samples):
      trials = 0
      while True:
        trials += 1
        idx = rng.choice(problem.n_var, SIZE, replace=False)
        cond = (
            problem.keeper[idx].sum() >= 1
            and problem.bowler[idx].sum() >= MIN_BOWLERS
            and problem.pure_bat[idx].sum() >= MIN_PURE_BAT
            and problem.pure_bowl[idx].sum() >= MIN_PURE_BOWL
        )
        if cond or trials > 1000:
          break
      X[i, idx] = True
    return X


class UnionCrossover(Crossover):

  def __init__(self, prob=0.9):
    super().__init__(2, 2, prob=prob)

  def _do(self, problem, X, **kwargs):
    X = np.asarray(X, dtype=bool)
    Y = X.copy()
    for i in range(X.shape[1]):
      a, b = X[0, i], X[1, i]
      both, either = np.flatnonzero(a & b), np.flatnonzero(a ^ b)
      need = SIZE - len(both)
      for j in range(2):
        Y[j, i] = False
        Y[j, i, both] = True
        if need > 0:
          Y[j, i, rng.choice(either, need, replace=False)] = True
    return Y


class FixSize(Repair):

  def _do(self, problem, X, **kwargs):
    X = np.asarray(X, dtype=bool).copy()
    for i in range(len(X)):
      on, off = np.flatnonzero(X[i]), np.flatnonzero(~X[i])
      k = len(on) - SIZE
      if k > 0:
        X[i, rng.choice(on, k, replace=False)] = False
      elif k < 0:
        X[i, rng.choice(off, -k, replace=False)] = True
    return X


class SwapMutation(Mutation):

  def _do(self, problem, X, **kwargs):
    X = np.asarray(X, dtype=bool).copy()
    for i in range(len(X)):
      on, off = np.flatnonzero(X[i]), np.flatnonzero(~X[i])
      k = rng.integers(1, 3)
      X[i, rng.choice(on, k, replace=False)] = False
      X[i, rng.choice(off, k, replace=False)] = True
    return X


prob = XIProblem(
    S.bat.values,
    S.bowl.values,
    S.keeper.values,
    S.bowler.values,
    S.pure_bat.values,
    S.pure_bowl.values,
)
algo = NSGA2(
    pop_size=100,
    sampling=RoleSampling(),
    crossover=UnionCrossover(prob=0.9),
    mutation=SwapMutation(),
    repair=FixSize(),
    eliminate_duplicates=True,
)
res = minimize(prob, algo, ('n_gen', 200), seed=1, verbose=False)

ok = np.asarray(res.opt.get('feasible')).ravel()
F, X = -res.opt.get('F')[ok], res.opt.get('X')[ok]


def knee(F):
  if len(F) < 3:
    return int(np.argmax(F.sum(1)))
  Fn = (F - F.min(0)) / np.where(np.ptp(F, 0) == 0, 1, np.ptp(F, 0))
  a, b = Fn[np.argmax(Fn[:, 0])], Fn[np.argmax(Fn[:, 1])]
  d = b - a
  if np.allclose(d, 0):
    return int(np.argmax(Fn.sum(1)))
  return int(
      np.argmax(
          np.abs(d[0] * (Fn[:, 1] - a[1]) - d[1] * (Fn[:, 0] - a[0]))
          / np.hypot(*d)
      )
  )


k = knee(F)
pred_pids = set(S.player_id.values[X[k]])
predicted_xi_names = sorted([names[p] for p in pred_pids])

print(
    '========================================================================='
)
print('   DEC 2026 INDIA vs SRI LANKA T20I: MODEL PREDICTED PLAYING XI         ')
print(
    '=========================================================================\n'
)

print(
    f'Target Match Date: {MATCH_DATE.strftime("%Y-%m-%d")} | Total Pool Size:'
    f' {len(pool)}\n'
)
print('Model Recommended 11 Players:')
for idx, name in enumerate(predicted_xi_names, 1):
  print(f'{idx:2d}. {name}')

   DEC 2026 INDIA vs SRI LANKA T20I: MODEL PREDICTED PLAYING XI         

Target Match Date: 2026-12-15 | Total Pool Size: 35

Model Recommended 11 Players:
 1. Abhishek Sharma
 2. Ashok Sharma
 3. B Kumar
 4. Ishan Kishan
 5. JJ Bumrah
 6. M Prasidh Krishna
 7. MP Yadav
 8. Nithish Kumar Reddy
 9. SV Samson
10. V Suryavanshi
11. Yash Thakur


In [43]:
import pickle
from collections import Counter
import numpy as np
import pandas as pd
from scipy.optimize import milp, LinearConstraint, Bounds

SIZE, MIN_BOWLERS = 11, 5
MIN_PURE_BAT, MIN_PURE_BOWL = 3, 2       # same values as Step 7
N_RANDOM = 200                           # random valid XIs per match
N_BOOT = 2000
rng = np.random.default_rng(7)

scores = pd.read_pickle('scores_step6.pkl')
P5 = pickle.load(open('pools_step5.pkl', 'rb'))
xi = P5['xi']
N7 = pickle.load(open('nsga_step7.pkl', 'rb'))['results']

# order of all India T20Is, for the "previous XI" and "last 10 matches" baselines
f = pd.read_pickle('feat_all.pkl')
ind = f[(f.team == 'India') & (f.src == 'intl') & (f.match_id != 'PHANTOM')]
seq = list(ind.groupby('match_id').date.first().sort_values().index)
pos = {m: i for i, m in enumerate(seq)}
del f


def best_xi(c, keeper, bowler, pb, pw):
    """exact XI maximising sum(c) under the same role constraints as the NSGA-II runs"""
    n = len(c)
    A = np.vstack([np.ones(n), keeper, bowler, pb, pw]).astype(float)
    lb = [SIZE, 1, MIN_BOWLERS, MIN_PURE_BAT, MIN_PURE_BOWL]
    ub = [SIZE, np.inf, np.inf, np.inf, np.inf]
    r = milp(-np.asarray(c, float), constraints=LinearConstraint(A, lb, ub),
             integrality=np.ones(n), bounds=Bounds(0, 1))
    return np.flatnonzero(r.x > 0.5) if r.status == 0 else None


def balanced_xi(bat, bowl, *roles):
    """equal weight on both objectives, each divided by its own exact maximum (ideal point)"""
    ib, iw = best_xi(bat, *roles), best_xi(bowl, *roles)
    if ib is None or iw is None:
        return None
    B, W = bat[ib].sum(), bowl[iw].sum()
    if B <= 0 or W <= 0:
        return None
    return best_xi(bat / B + bowl / W, *roles)


def random_valid(n, roles, k=N_RANDOM, max_trials=50000):
    keeper, bowler, pb, pw = roles
    out, trials = [], 0
    while len(out) < k and trials < max_trials:
        trials += 1
        idx = rng.choice(n, SIZE, replace=False)
        if (keeper[idx].sum() >= 1 and bowler[idx].sum() >= MIN_BOWLERS
                and pb[idx].sum() >= MIN_PURE_BAT and pw[idx].sum() >= MIN_PURE_BOWL):
            out.append(idx)
    return out


rows = []
for (m, mid), S in scores.groupby(['months', 'match_id'], sort=False):
    if (m, mid) not in N7:
        continue
    S = S.reset_index(drop=True)
    pid = S.player_id.values
    real = xi[mid]
    keeper = S.keeper.values.astype(float)
    if keeper.sum() == 0:                                   # same fallback as Step 7
        keeper = (S.p_bat.values >= 0.5).astype(float)
    roles = (keeper, S.bowler.values.astype(float), S.pure_bat.values.astype(float),
             S.pure_bowl.values.astype(float))

    ov = lambda ids: len(set(ids) & real)
    r = dict(match_id=mid, pool=len(S), ceiling=len(set(pid) & real))
    r['NSGA-II knee (ML scores)'] = ov(N7[(m, mid)]['knee_xi'])

    for name, bat, bowl in [('Balanced XI (ML scores)', S.bat.values, S.bowl.values),
                            ('Balanced XI (rolling form)', S.bat_form.values, S.bowl_form.values)]:
        idx = balanced_xi(bat, bowl, *roles)
        r[name] = ov(pid[idx]) if idx is not None else np.nan

    i = pos[mid]
    r['Previous match XI'] = ov(xi[seq[i - 1]]) if i >= 1 else np.nan
    cnt = Counter(p for mm in seq[max(0, i - 10):i] for p in xi[mm])
    r['Top 11 of last 10 matches'] = ov([p for p, _ in sorted(cnt.items(), key=lambda t: (-t[1], str(t[0])))[:SIZE]])

    rv = random_valid(len(S), roles)
    r['Random valid XI (expected)'] = np.mean([ov(pid[x]) for x in rv]) if rv else np.nan
    rows.append(r)

D = pd.DataFrame(rows)
methods = [c for c in D.columns if c not in ('match_id', 'pool', 'ceiling')]
print(f'{len(D)} matches | mean pool {D.pool.mean():.1f} | max possible for pool-based methods '
      f'(real XI players inside pool) {D.ceiling.mean():.2f}\n')


def boot(x, n=N_BOOT):
    x = np.asarray(x, float)
    x = x[~np.isnan(x)]
    b = rng.choice(x, size=(n, len(x)), replace=True).mean(1)
    return x.mean(), np.percentile(b, 2.5), np.percentile(b, 97.5)


tab = pd.DataFrame([dict(method=c, mean=boot(D[c])[0], lo=boot(D[c])[1], hi=boot(D[c])[2]) for c in methods])
print(tab.round(2).to_string(index=False))

print('\npaired differences (mean, 95% bootstrap CI over matches)')
for a, b in [('NSGA-II knee (ML scores)', 'Random valid XI (expected)'),
             ('NSGA-II knee (ML scores)', 'Previous match XI'),
             ('Balanced XI (ML scores)', 'Balanced XI (rolling form)'),
             ('NSGA-II knee (ML scores)', 'Balanced XI (ML scores)')]:
    mu, lo, hi = boot(D[a] - D[b])
    print(f'  {a} - {b}: {mu:+.2f} [{lo:+.2f}, {hi:+.2f}]')

pickle.dump(dict(per_match=D, table=tab), open('baselines_step8.pkl', 'wb'))
print('\nsaved baselines_step8.pkl')

55 matches | mean pool 36.1 | max possible for pool-based methods (real XI players inside pool) 10.82

                    method  mean   lo   hi
  NSGA-II knee (ML scores)  2.91 2.65 3.18
   Balanced XI (ML scores)  3.29 3.07 3.53
Balanced XI (rolling form)  3.09 2.65 3.53
         Previous match XI  9.29 8.95 9.60
 Top 11 of last 10 matches  8.15 7.78 8.49
Random valid XI (expected)  3.33 3.23 3.44

paired differences (mean, 95% bootstrap CI over matches)
  NSGA-II knee (ML scores) - Random valid XI (expected): -0.42 [-0.63, -0.20]
  NSGA-II knee (ML scores) - Previous match XI: -6.38 [-6.76, -5.98]
  Balanced XI (ML scores) - Balanced XI (rolling form): +0.20 [-0.18, +0.55]
  NSGA-II knee (ML scores) - Balanced XI (ML scores): -0.38 [-0.58, -0.20]

saved baselines_step8.pkl


In [44]:
import pickle
from collections import Counter
import numpy as np
import pandas as pd
from sklearn.ensemble import HistGradientBoostingClassifier
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import roc_auc_score
from sklearn.pipeline import make_pipeline
from sklearn.preprocessing import StandardScaler

# This cell reuses build_pool, ind and xi from Step 5, so run Step 5 in the same session first.
TRAIN_START, TEST_START = pd.Timestamp('2021-01-01'), pd.Timestamp('2025-01-01')

f = pd.read_pickle('feat_all.pkl')
fs = f.sort_values(['player_id', 'date']).reset_index(drop=True)
pdates_all = {p: g.date.values for p, g in fs.groupby('player_id')}
first = fs.groupby('player_id').head(1)
base_idx = dict(zip(first.player_id, first.index))
asof_row = lambda p, d: base_idx[p] + np.searchsorted(pdates_all[p], np.datetime64(d), 'left')

# India T20I order and per-player India appearance dates
order = ind.groupby('match_id').date.first().sort_values()
seq = list(order.index)
pos = {m: i for i, m in enumerate(seq)}
india_dates = {p: np.sort(g.date.values) for p, g in ind.groupby('player_id')}


def selection_rows(mid, date):
    """one row per pool player: what a selector could see before the match, and whether he played"""
    pool, grp_a, grp_b = build_pool(date, 12)
    i = pos[mid]
    prev = xi[seq[i - 1]] if i >= 1 else set()
    last5 = Counter(p for mm in seq[max(0, i - 5):i] for p in xi[mm])
    last10 = Counter(p for mm in seq[max(0, i - 10):i] for p in xi[mm])
    X = fs.loc[[asof_row(p, date) for p in pool]].reset_index(drop=True)
    rows = []
    for k, p in enumerate(pool):
        d = india_dates.get(p, np.array([], dtype='datetime64[ns]'))
        n_ind = int(np.searchsorted(d, np.datetime64(date), 'left'))
        days = (date - pd.Timestamp(d[n_ind - 1])).days if n_ind > 0 else 730
        rows.append(dict(
            match_id=mid, date=date, player_id=p,
            in_prev=int(p in prev), frac5=last5[p] / 5, frac10=last10[p] / 10,
            n_india=n_ind, days_india=min(days, 730), ipl_entrant=int(p not in grp_a),
            matches_prev=X.matches_prev[k], p_bat=X.p_bat_15[k], p_bowl=X.p_bowl_15[k],
            bat_form=X.bat_runs_15[k], bowl_form=X.bowl_wickets_15[k],
            keeper=int(X.stump_prev[k] >= 2), played=int(p in xi[mid])))
    return rows


FEATS = ['in_prev', 'frac5', 'frac10', 'n_india', 'days_india', 'ipl_entrant',
         'matches_prev', 'p_bat', 'p_bowl', 'bat_form', 'bowl_form', 'keeper']

mids = [m for m in seq if order[m] >= TRAIN_START]
data = pd.DataFrame([r for m in mids for r in selection_rows(m, order[m])])
tr, te = data[data.date < TEST_START], data[data.date >= TEST_START]
print(f'matches: train {tr.match_id.nunique()} | test {te.match_id.nunique()} | rows: train {len(tr):,} | test {len(te):,}')
print(f'share of pool players who played: train {tr.played.mean():.3f} | test {te.played.mean():.3f}')

models = {
    'logistic': make_pipeline(StandardScaler(), LogisticRegression(C=1.0, max_iter=2000)),
    'grad. boosting': HistGradientBoostingClassifier(max_depth=3, learning_rate=0.05, max_iter=200, random_state=0),
}


def top11_overlap(df, p):
    out = []
    for mid, g in df.assign(p=p).groupby('match_id'):
        top = g.sort_values('p', ascending=False).head(11)
        out.append(top.played.sum())
    return np.array(out)


print('\ntest period (2025+), pool players ranked by P(selected)')
res = {}
for name, mdl in models.items():
    mdl.fit(tr[FEATS], tr.played)
    p = mdl.predict_proba(te[FEATS])[:, 1]
    res[name] = p
    ov = top11_overlap(te, p)
    print(f'  {name:15s} AUC {roc_auc_score(te.played, p):.3f} | top-11 overlap {ov.mean():.2f} of 11')
prev_ov = te.groupby('match_id').apply(lambda g: g[g.in_prev == 1].played.sum()).mean()
print(f'  previous XI in pool (same pool): {prev_ov:.2f}  | pool-ceiling {te.groupby("match_id").played.sum().mean():.2f}')

lr = models['logistic']
coef = pd.Series(lr[-1].coef_[0], index=FEATS).sort_values()
print('\nlogistic coefficients (standardised features)')
print(coef.round(2).to_string())

# probabilities for every pool player (used as a third objective in the next step)
data['p_select'] = np.nan
data.loc[te.index, 'p_select'] = res['logistic']
data.loc[tr.index, 'p_select'] = lr.predict_proba(tr[FEATS])[:, 1]       # in-sample, for reference only
pickle.dump(dict(data=data, feats=FEATS, model=lr), open('select_step9.pkl', 'wb'))
print('\nsaved select_step9.pkl', data.shape)

matches: train 98 | test 55 | rows: train 4,490 | test 1,983
share of pool players who played: train 0.233 | test 0.300

test period (2025+), pool players ranked by P(selected)
  logistic        AUC 0.951 | top-11 overlap 9.29 of 11
  grad. boosting  AUC 0.942 | top-11 overlap 9.29 of 11
  previous XI in pool (same pool): 9.29  | pool-ceiling 10.82

logistic coefficients (standardised features)
days_india     -1.32
keeper         -0.01
frac5          -0.00
n_india         0.03
matches_prev    0.07
frac10          0.15
bowl_form       0.21
bat_form        0.23
p_bowl          0.26
p_bat           0.34
ipl_entrant     0.51
in_prev         1.24

saved select_step9.pkl (6473, 17)


C:\Users\Ashutosh Kumar\AppData\Local\Temp\ipykernel_8212\2765830860.py:82: FutureWarning: DataFrameGroupBy.apply operated on the grouping columns. This behavior is deprecated, and in a future version of pandas the grouping columns will be excluded from the operation. Either pass `include_groups=False` to exclude the groupings or explicitly select the grouping columns after groupby to silence this warning.
  prev_ov = te.groupby('match_id').apply(lambda g: g[g.in_prev == 1].played.sum()).mean()


In [45]:
import pickle
import numpy as np
import pandas as pd

# Step 7 results, Step 9 selection table, Step 5 real XIs
res7 = pickle.load(open('nsga_step7.pkl', 'rb'))['results']
data = pickle.load(open('select_step9.pkl', 'rb'))['data']
xi = pickle.load(open('pools_step5.pkl', 'rb'))['xi']

# IPL entrants (not an India T20I player in the last 12 months) per match
entrants = data[data.ipl_entrant == 1].groupby('match_id').player_id.agg(set)
in_data = data.groupby('match_id').player_id.agg(set)

rows = []
for (m, mid), r in res7.items():
    ent = entrants.get(mid, set())
    knee = set(r['knee_xi'])
    real = set(xi[mid])

    # every knee player comes from the pool, so it must be present in data
    assert knee <= in_data[mid], f'knee player missing from data in {mid}'

    rows.append(dict(
        match_id=mid,
        knee_entrants=len(knee & ent),
        real_entrants=len(real & ent),
        real_not_in_pool=len(real - in_data[mid]),   # real XI players the pool never had
    ))

D = pd.DataFrame(rows)
print(f'matches: {len(D)}')
print(f'IPL entrants per XI | knee: {D.knee_entrants.mean():.2f} | real: {D.real_entrants.mean():.2f}')
print(f'matches with zero entrants | knee: {(D.knee_entrants == 0).mean():.2f} | real: {(D.real_entrants == 0).mean():.2f}')
print(f'real XI players outside the pool (counted as non-entrants): {D.real_not_in_pool.sum()}')

matches: 55
IPL entrants per XI | knee: 4.16 | real: 0.15
matches with zero entrants | knee: 0.00 | real: 0.89
real XI players outside the pool (counted as non-entrants): 11


In [46]:
import pickle
import time
import numpy as np
import pandas as pd
from scipy.optimize import milp, LinearConstraint, Bounds
from pymoo.algorithms.moo.nsga2 import NSGA2
from pymoo.core.crossover import Crossover
from pymoo.core.mutation import Mutation
from pymoo.core.problem import Problem
from pymoo.core.repair import Repair
from pymoo.core.sampling import Sampling
from pymoo.optimize import minimize

SIZE, MIN_BOWLERS = 11, 5
MIN_PURE_BAT, MIN_PURE_BOWL = 3, 2      # same as Step 7
POP, GEN, SEED = 100, 200, 1
DRY_RUN = 5                             # number of matches for a quick check, None = all 55
TOP_N = 20                              # pool size for the filter variant
LAMBDA_MAIN, LAMBDAS = 1.0, (0.5, 1.0, 2.0)
N_BOOT = 2000
rng = np.random.default_rng(SEED)

scores = pd.read_pickle('scores_step6.pkl')
scores = scores[scores.months == 12]
sel = pickle.load(open('select_step9.pkl', 'rb'))['data'][['match_id', 'player_id', 'p_select', 'ipl_entrant']]
scores = scores.merge(sel, on=['match_id', 'player_id'], how='left')
assert scores.p_select.notna().all(), 'pool in Step 6 and Step 9 differ'

xi = pickle.load(open('pools_step5.pkl', 'rb'))['xi']
N7 = pickle.load(open('nsga_step7.pkl', 'rb'))['results']
D8 = pickle.load(open('baselines_step8.pkl', 'rb'))['per_match'].set_index('match_id')


# ---------- exact solver ----------
def best_xi(c, keeper, bowler, pb, pw):
    """exact XI maximising sum(c) under the role constraints, None if infeasible"""
    n = len(c)
    A = np.vstack([np.ones(n), keeper, bowler, pb, pw]).astype(float)
    lb = [SIZE, 1, MIN_BOWLERS, MIN_PURE_BAT, MIN_PURE_BOWL]
    ub = [SIZE, np.inf, np.inf, np.inf, np.inf]
    r = milp(-np.asarray(c, float), constraints=LinearConstraint(A, lb, ub),
             integrality=np.ones(n), bounds=Bounds(0, 1))
    return np.flatnonzero(r.x > 0.5) if r.status == 0 else None


# ---------- NSGA-II pieces (same operators as Step 7, objectives made generic) ----------
class XIProblem(Problem):
    def __init__(self, objs, keeper, bowler, pb, pw):
        super().__init__(n_var=len(keeper), n_obj=len(objs), n_ieq_constr=4, xl=0, xu=1, vtype=bool)
        self.objs = np.column_stack(objs)
        self.keeper, self.bowler, self.pb, self.pw = keeper, bowler, pb, pw

    def _evaluate(self, X, out, *args, **kwargs):
        X = X.astype(float)
        out['F'] = -(X @ self.objs)
        out['G'] = np.column_stack([1 - X @ self.keeper, MIN_BOWLERS - X @ self.bowler,
                                    MIN_PURE_BAT - X @ self.pb, MIN_PURE_BOWL - X @ self.pw])


class RoleSampling(Sampling):
    def _do(self, problem, n_samples, **kwargs):
        X = np.zeros((n_samples, problem.n_var), dtype=bool)
        for i in range(n_samples):
            for _ in range(1000):
                idx = rng.choice(problem.n_var, SIZE, replace=False)
                if (problem.keeper[idx].sum() >= 1 and problem.bowler[idx].sum() >= MIN_BOWLERS
                        and problem.pb[idx].sum() >= MIN_PURE_BAT and problem.pw[idx].sum() >= MIN_PURE_BOWL):
                    break
            X[i, idx] = True
        return X


class UnionCrossover(Crossover):
    def __init__(self, prob=0.9):
        super().__init__(2, 2, prob=prob)

    def _do(self, problem, X, **kwargs):
        X = np.asarray(X, dtype=bool)
        Y = X.copy()
        for i in range(X.shape[1]):
            a, b = X[0, i], X[1, i]
            both, either = np.flatnonzero(a & b), np.flatnonzero(a ^ b)
            need = SIZE - len(both)
            for j in range(2):
                Y[j, i] = False
                Y[j, i, both] = True
                if need > 0:
                    Y[j, i, rng.choice(either, need, replace=False)] = True
        return Y


class FixSize(Repair):
    def _do(self, problem, X, **kwargs):
        X = np.asarray(X, dtype=bool).copy()
        for i in range(len(X)):
            on, off = np.flatnonzero(X[i]), np.flatnonzero(~X[i])
            k = len(on) - SIZE
            if k > 0:
                X[i, rng.choice(on, k, replace=False)] = False
            elif k < 0:
                X[i, rng.choice(off, -k, replace=False)] = True
        return X


class SwapMutation(Mutation):
    def _do(self, problem, X, **kwargs):
        X = np.asarray(X, dtype=bool).copy()
        for i in range(len(X)):
            on, off = np.flatnonzero(X[i]), np.flatnonzero(~X[i])
            k = rng.integers(1, 3)
            X[i, rng.choice(on, k, replace=False)] = False
            X[i, rng.choice(off, k, replace=False)] = True
        return X


def knee2(F):
    """2-objective knee: farthest point from the line joining the two extremes (as in Step 7)"""
    if len(F) < 3:
        return int(np.argmax(F.sum(1)))
    rngs = np.where(np.ptp(F, 0) == 0, 1, np.ptp(F, 0))
    Fn = (F - F.min(0)) / rngs
    a, b = Fn[np.argmax(Fn[:, 0])], Fn[np.argmax(Fn[:, 1])]
    d = b - a
    if np.allclose(d, 0):
        return int(np.argmax(Fn.sum(1)))
    dist = np.abs(d[0] * (Fn[:, 1] - a[1]) - d[1] * (Fn[:, 0] - a[0])) / np.hypot(*d)
    return int(np.argmax(dist))


def knee_ideal(F):
    """3-objective pick: closest point to the ideal corner after min-max normalisation"""
    rngs = np.where(np.ptp(F, 0) == 0, 1, np.ptp(F, 0))
    Fn = (F - F.min(0)) / rngs
    return int(np.argmin(np.linalg.norm(1 - Fn, axis=1)))


def run_nsga(objs, roles, n_obj):
    prob = XIProblem(objs, *roles)
    algo = NSGA2(pop_size=POP, sampling=RoleSampling(), crossover=UnionCrossover(prob=0.9),
                 mutation=SwapMutation(), repair=FixSize(), eliminate_duplicates=True)
    res = minimize(prob, algo, ('n_gen', GEN), seed=SEED, verbose=False)
    ok = np.asarray(res.opt.get('feasible')).ravel()
    if not ok.any():
        return None
    F, X = -res.opt.get('F')[ok], res.opt.get('X')[ok]
    k = knee2(F) if n_obj == 2 else knee_ideal(F)
    return np.flatnonzero(X[k])


# ---------- main loop ----------
mids = list(scores.match_id.unique())
if DRY_RUN:
    mids = mids[:DRY_RUN]

rows, n_used = [], {}
t0 = time.time()
for i, mid in enumerate(mids, 1):
    S = scores[scores.match_id == mid].reset_index(drop=True)
    pid, real = S.player_id.values, set(xi[mid])
    keeper = S.keeper.values.astype(float)
    if keeper.sum() == 0:                                   # same fallback as Step 7
        keeper = (S.p_bat.values >= 0.5).astype(float)
    roles = (keeper, S.bowler.values.astype(float), S.pure_bat.values.astype(float),
             S.pure_bowl.values.astype(float))
    bat, bowl, ps = S.bat.values, S.bowl.values, S.p_select.values

    picks = {'Step 7 knee (2-obj)': np.flatnonzero(np.isin(pid, list(N7[(12, mid)]['knee_xi'])))}

    # (i) third objective
    picks['(i) 3-obj NSGA-II'] = run_nsga([bat, bowl, ps], roles, 3)

    # (ii) weighted score, each term divided by its exact maximum
    B = bat[best_xi(bat, *roles)].sum()
    W = bowl[best_xi(bowl, *roles)].sum()
    P = ps[best_xi(ps, *roles)].sum()
    for lam in LAMBDAS:
        picks[f'(ii) weighted MILP lam={lam}'] = best_xi(bat / B + bowl / W + lam * ps / P, *roles)

    # (iii) keep top-N by p_select, then the usual 2-objective run
    n = TOP_N
    while True:
        keep = np.argsort(-ps)[:n]
        sub_roles = tuple(r[keep] for r in roles)
        if best_xi(np.ones(len(keep)), *sub_roles) is not None or n >= len(S):
            break
        n += 5
    n_used[mid] = n
    sub = run_nsga([bat[keep], bowl[keep]], sub_roles, 2)
    picks['(iii) top-N filter + 2-obj'] = keep[sub] if sub is not None else None

    for name, idx in picks.items():
        if idx is None:
            rows.append(dict(match_id=mid, method=name, overlap=np.nan, entrants=np.nan,
                             f1=np.nan, f2=np.nan, f3=np.nan))
            continue
        rows.append(dict(match_id=mid, method=name, overlap=len(set(pid[idx]) & real),
                         entrants=S.ipl_entrant.values[idx].sum(),
                         f1=bat[idx].sum(), f2=bowl[idx].sum(), f3=ps[idx].sum()))
    el = time.time() - t0
    print(f'[{i}/{len(mids)}] elapsed {el / 60:.1f}m | ETA {el / i * (len(mids) - i) / 60:.1f}m | pool {len(S)}')

L = pd.DataFrame(rows)

# overlap-only baselines from Step 8, same matches
for name in ('Previous match XI', 'Random valid XI (expected)'):
    L = pd.concat([L, pd.DataFrame(dict(match_id=mids, method=name,
                                        overlap=D8.loc[mids, name].values))], ignore_index=True)

# f1 / f2 as a share of the Step 7 knee, per match
ref = L[L.method == 'Step 7 knee (2-obj)'].set_index('match_id')
L['f1_rel'] = L.f1 / L.match_id.map(ref.f1)
L['f2_rel'] = L.f2 / L.match_id.map(ref.f2)


def boot(x, n=N_BOOT):
    x = np.asarray(x, float)
    x = x[~np.isnan(x)]
    b = rng.choice(x, size=(n, len(x)), replace=True).mean(1)
    return x.mean(), np.percentile(b, 2.5), np.percentile(b, 97.5)


order = ['Previous match XI', 'Random valid XI (expected)', 'Step 7 knee (2-obj)', '(i) 3-obj NSGA-II'] \
        + [f'(ii) weighted MILP lam={l}' for l in LAMBDAS] + ['(iii) top-N filter + 2-obj']
tab = []
for m in order:
    g = L[L.method == m]
    mu, lo, hi = boot(g.overlap)
    tab.append(dict(method=m, overlap=mu, lo=lo, hi=hi, entrants=g.entrants.mean(),
                    f1_rel=g.f1_rel.mean(), f2_rel=g.f2_rel.mean(), f3=g.f3.mean()))
print(f'\nmatches: {len(mids)} | filter N used (mean): {np.mean(list(n_used.values())):.1f}')
print(pd.DataFrame(tab).round(2).to_string(index=False))

W_ = L.pivot(index='match_id', columns='method', values='overlap')
print('\npaired overlap differences (mean, 95% bootstrap CI over matches)')
for a, b in [('(i) 3-obj NSGA-II', 'Step 7 knee (2-obj)'),
             (f'(ii) weighted MILP lam={LAMBDA_MAIN}', 'Step 7 knee (2-obj)'),
             ('(iii) top-N filter + 2-obj', 'Step 7 knee (2-obj)'),
             (f'(ii) weighted MILP lam={LAMBDA_MAIN}', 'Previous match XI'),
             ('(iii) top-N filter + 2-obj', 'Previous match XI')]:
    mu, lo, hi = boot(W_[a] - W_[b])
    print(f'  {a} - {b}: {mu:+.2f} [{lo:+.2f}, {hi:+.2f}]')

pickle.dump(dict(long=L, table=pd.DataFrame(tab), n_used=n_used), open('step10.pkl', 'wb'))
print('\nsaved step10.pkl')

[1/5] elapsed 0.2m | ETA 0.9m | pool 43
[2/5] elapsed 0.4m | ETA 0.6m | pool 43
[3/5] elapsed 0.6m | ETA 0.4m | pool 43
[4/5] elapsed 0.9m | ETA 0.2m | pool 44
[5/5] elapsed 1.1m | ETA 0.0m | pool 45

matches: 5 | filter N used (mean): 20.0
                    method  overlap   lo    hi  entrants  f1_rel  f2_rel   f3
         Previous match XI     9.40 8.60 10.00       NaN     NaN     NaN  NaN
Random valid XI (expected)     2.78 2.72  2.86       NaN     NaN     NaN  NaN
       Step 7 knee (2-obj)     3.20 2.40  4.20       3.2    1.00    1.00 3.36
         (i) 3-obj NSGA-II     8.20 8.00  8.60       1.2    1.24    0.80 7.41
(ii) weighted MILP lam=0.5     8.80 8.40  9.00       0.8    1.15    0.84 8.27
(ii) weighted MILP lam=1.0     9.20 9.00  9.60       0.0    1.30    0.73 8.58
(ii) weighted MILP lam=2.0     9.20 8.60  9.80       0.0    1.26    0.68 8.99
(iii) top-N filter + 2-obj     7.20 7.00  7.60       0.0    1.36    0.75 6.57

paired overlap differences (mean, 95% bootstrap CI over 